# Data Cleaning: Before & After

This notebook is the explanatory layer for a reusable retail-order cleaning pipeline. The production logic lives in `scripts/clean_retail_orders.py`; the notebook profiles the source, executes the pipeline, shows the review queue, and verifies the output contracts.

## 1. Environment and reusable pipeline

The project path is resolved from either the repository root, project folder, or `notebooks/` folder. All transformation functions are imported from the reusable script so the notebook and command-line workflow share one implementation.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "raw" / "dirty_retail_orders.csv").exists():
    if (PROJECT_DIR.parent / "raw" / "dirty_retail_orders.csv").exists():
        PROJECT_DIR = PROJECT_DIR.parent
    elif (PROJECT_DIR / "04-data-cleaning-before-after" / "raw" / "dirty_retail_orders.csv").exists():
        PROJECT_DIR = PROJECT_DIR / "04-data-cleaning-before-after"
    else:
        raise FileNotFoundError("Could not locate 04-data-cleaning-before-after/raw/dirty_retail_orders.csv")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from scripts.clean_retail_orders import load_raw_orders, run_pipeline

## 2. Raw source profile

Every source column is loaded as text. This preserves malformed dates, currency symbols, written quantities, missing values, and inconsistent labels for audit before conversion.

In [2]:
raw_orders = load_raw_orders(PROJECT_DIR / "raw" / "dirty_retail_orders.csv")
raw_profile = pd.DataFrame({
    "metric": ["rows", "columns", "logical source files", "duplicated order IDs", "blank cells"],
    "value": [
        len(raw_orders),
        len(raw_orders.columns),
        raw_orders["source_file"].nunique(),
        raw_orders.loc[raw_orders["order_id"].duplicated(keep=False), "order_id"].nunique(),
        int(raw_orders.eq("").sum().sum()),
    ],
})
display(raw_profile)

              metric  value
                rows     63
             columns     22
logical source files      4
duplicated order IDs      2
         blank cells     57

## 3. Execute the raw-to-clean pipeline

The pipeline standardizes text and dates, parses numeric values, calculates sales fields, assigns row-level quality issues, resolves duplicate order IDs by source priority, validates contracts, and writes full, publishable, and review outputs.

In [3]:
result = run_pipeline(PROJECT_DIR, write_output_files=True)
clean_orders = result["clean"]
publishable_orders = result["publishable"]
review_orders = result["review"]

## 4. Before-and-after quality metrics

The clean output retains review-worthy business events rather than silently deleting them. Publishable records and the review queue are written separately for safe downstream use.

In [4]:
display(result["quality_summary"])

                     metric   value
                   raw_rows   63.00
                 clean_rows   61.00
removed_duplicate_order_ids    2.00
              valid_records   50.00
   records_requiring_review   11.00
             valid_rate_pct   81.97
            review_rate_pct   18.03
     valid_record_net_sales 4003.70

## 5. Issue frequency

Issue counts identify the controls that fired. A single order can contribute more than one issue, which is why issue counts are not expected to equal the number of review records.

In [5]:
display(result["issue_summary"])

              quality_issue  records
ambiguous_order_date_format        1
              invalid_email        1
         invalid_order_date        1
           invalid_quantity        1
         invalid_unit_price        1
           missing_category        1
        missing_customer_id        1
              missing_email        1
        missing_product_sku        1
          negative_discount        1
        negative_unit_price        1
      non_positive_quantity        1

## 6. Review queue

These records remain traceable in the full clean dataset but are excluded from the publishable subset until an owner resolves the flagged issue.

In [6]:
display(
    review_orders[["order_id", "source_file", "record_quality_status", "quality_issues"]]
    .sort_values("order_id")
    .reset_index(drop=True)
)

order_id    source_file record_quality_status                         quality_issues
ORD-1004 jan_export.csv                review                          invalid_email
ORD-1018 jan_export.csv                review                       invalid_quantity
ORD-1023 feb_export.csv                review            ambiguous_order_date_format
ORD-1026 feb_export.csv                review                      negative_discount
ORD-1027 feb_export.csv                review                  non_positive_quantity
ORD-1028 feb_export.csv                review                     invalid_unit_price
ORD-1039 feb_export.csv                review                     invalid_order_date
ORD-1044 mar_export.csv                review                          missing_email
ORD-1048 mar_export.csv                review                    negative_unit_price
ORD-1052 mar_export.csv                review missing_product_sku | missing_category
ORD-1061 mar_export.csv                review                    

## 7. Concrete before-and-after examples

Selected records show how mixed formats become canonical analytical fields while the original business key remains stable.

In [7]:
sample_ids = ["ORD-1002", "ORD-1007", "ORD-1018", "ORD-1023"]
raw_sample = raw_orders.loc[raw_orders["order_id"].isin(sample_ids), [
    "order_id", "order_date", "country", "category", "quantity", "unit_price", "discount", "source_file"
]]
clean_sample = clean_orders.loc[clean_orders["order_id"].isin(sample_ids), [
    "order_id", "order_date", "country", "category", "quantity", "unit_price", "discount_rate", "source_file", "record_quality_status"
]]
display(raw_sample.sort_values(["order_id", "source_file"]).reset_index(drop=True))
display(clean_sample.sort_values("order_id").reset_index(drop=True))

order_id order_date       country    category quantity unit_price discount           source_file
ORD-1002 01/05/2025 United States Electronics        1     $18.50       5%        jan_export.csv
ORD-1002 2025-01-05           USA Electronics        1      18.50     0.05 manual_adjustment.csv
ORD-1007 13/01/2025           USA     Fitness        1      29.99      15%        jan_export.csv
ORD-1018 2025-01-26      Honduras     Apparel               49.99        0        jan_export.csv
ORD-1023 02-03-2025      Honduras Electronics        3       9.99        0        feb_export.csv

order_id order_date       country    category quantity unit_price  discount_rate           source_file record_quality_status
ORD-1002 2025-01-05 United States Electronics      1.0       18.5           0.05 manual_adjustment.csv                 valid
ORD-1007 2025-01-13 United States     Fitness      1.0      29.99           0.15        jan_export.csv                 valid
ORD-1018 2025-01-26      Honduras     Apparel     <NA>      49.99           0.00        jan_export.csv                review
ORD-1023 2025-02-03      Honduras Electronics      3.0       9.99           0.00        feb_export.csv                review

## 8. Validation contracts

The assertions below protect uniqueness, row counts, status values, output separation, and expected artifact creation.

In [8]:
assert len(raw_orders) == 63
assert len(clean_orders) == 61
assert not clean_orders["order_id"].duplicated().any()
assert len(publishable_orders) == 50
assert len(review_orders) == 11
assert set(clean_orders["record_quality_status"]) == {"valid", "review"}
assert set(publishable_orders["record_quality_status"]) == {"valid"}
assert set(review_orders["record_quality_status"]) == {"review"}
assert all(path.exists() for path in result["output_paths"].values())
print("All pipeline validation contracts passed.")

All pipeline validation contracts passed.


## 9. Generated deliverables

In [9]:
output_inventory = pd.DataFrame([
    {"artifact": name, "path": path.relative_to(PROJECT_DIR).as_posix(), "bytes": path.stat().st_size}
    for name, path in result["output_paths"].items()
])
display(output_inventory)

       artifact                                  path  bytes
          clean       cleaned/retail_orders_clean.csv  13925
    publishable cleaned/retail_orders_publishable.csv  11316
         review      cleaned/retail_orders_review.csv   2908
quality_summary      cleaned/data_quality_summary.csv    206
 quality_issues       cleaned/data_quality_issues.csv    287
   cleaning_log                  docs/cleaning-log.md    833